In [ ]:
Built a GitHub-ready Generative AI platform for inverse materials design of solid-state battery electrolytes using a conditional VAE, surrogate neural prediction, uncertainty-aware screening, and Flask-based API/dashboard deployment.

In [ ]:
GenMatDesign-AI/
├── app.py
├── requirements.txt
├── Dockerfile
├── README.md
├── LICENSE
├── .gitignore
├── artifacts/
│   └── .gitkeep
├── static/
│   └── preview.png
└── docs/
    └── architecture.md

In [ ]:
# GenMatDesign-AI
Generative AI for Inverse Materials Design of Solid-State Battery Electrolytes

## Overview
GenMatDesign-AI is a deployable Flask-based inverse materials design platform that combines:

- Conditional Variational Autoencoder (CVAE) for generative candidate design
- Surrogate neural network for property prediction
- Uncertainty-aware screening using MC dropout
- Token-level explanation for generated electrolyte sequences
- Web dashboard and REST API deployment

Instead of testing a material and then measuring properties, this system accepts desired target properties and generates candidate electrolyte compositions likely to satisfy them.

## Domain
This prototype is focused on **solid-state battery electrolytes** represented as token sequences such as:

- `PEO-LiTFSI-SN-Al2O3`
- `PEGDA-LiFSI-EC-LLZO`

## Features
- Built-in internal electrolyte dataset
- No CSV upload required
- Train from internal library with one click
- Predict properties of a given electrolyte sequence
- Generate new candidates from desired target properties
- Rank candidates by closeness to target
- Estimate predictive uncertainty
- Token-importance style interpretation
- Web UI + REST API

## Tech stack
- Python
- Flask
- PyTorch
- NumPy
- Pandas
- Matplotlib

## Repository structure
```text
GenMatDesign-AI/
├── app.py
├── requirements.txt
├── Dockerfile
├── README.md
├── LICENSE
├── .gitignore
├── artifacts/
│   └── .gitkeep
├── static/
│   └── preview.png
└── docs/
    └── architecture.md

In [8]:
#!/usr/bin/env python3
"""
Generative AI for Inverse Materials Design
Single-file Flask app
No CSV upload
No external dataset required

Domain:
- Solid-state battery electrolytes

This app:
1. Uses an internal built-in dataset
2. Trains a Conditional VAE generator
3. Trains a surrogate regressor
4. Generates candidate electrolyte sequences from desired properties
5. Exposes both web dashboard and REST API
"""

import os
import io
import json
import math
import base64
import random
import traceback
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd
from flask import Flask, request, jsonify, render_template_string

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Reproducibility
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

TARGET_COLUMNS = [
    "ionic_conductivity",
    "stability_window",
    "glass_transition_temp",
    "stability_score",
]

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

MAX_LEN = 14
EMBED_DIM = 64
HIDDEN_DIM = 128
LATENT_DIM = 32
BATCH_SIZE = 32
EPOCHS_CVAE = 50
EPOCHS_SURROGATE = 100
LR = 1e-3
MC_DROPOUT_RUNS = 15


# ============================================================
# Built-in internal dataset
# No upload needed
# ============================================================
def build_internal_dataset() -> pd.DataFrame:
    rows = [
        {"material_id":"M1",  "sequence":"PEO-LiTFSI-SN-Al2O3",   "ionic_conductivity":1.20, "stability_window":4.80, "glass_transition_temp":38, "stability_score":0.82},
        {"material_id":"M2",  "sequence":"PEGDA-LiFSI-EC-LLZO",   "ionic_conductivity":1.85, "stability_window":5.10, "glass_transition_temp":29, "stability_score":0.90},
        {"material_id":"M3",  "sequence":"PEO-LiFSI-PC-TiO2",     "ionic_conductivity":1.10, "stability_window":4.60, "glass_transition_temp":42, "stability_score":0.78},
        {"material_id":"M4",  "sequence":"PVDF-LiTFSI-SN-LATP",   "ionic_conductivity":1.55, "stability_window":5.30, "glass_transition_temp":31, "stability_score":0.87},
        {"material_id":"M5",  "sequence":"PEO-LiTFSI-EC-ZrO2",    "ionic_conductivity":1.30, "stability_window":4.90, "glass_transition_temp":36, "stability_score":0.83},
        {"material_id":"M6",  "sequence":"PEG-LiFSI-SN-Al2O3",    "ionic_conductivity":1.72, "stability_window":5.00, "glass_transition_temp":28, "stability_score":0.89},
        {"material_id":"M7",  "sequence":"PEO-LiBOB-PC-SiO2",     "ionic_conductivity":0.95, "stability_window":4.40, "glass_transition_temp":44, "stability_score":0.74},
        {"material_id":"M8",  "sequence":"PCL-LiTFSI-SN-LLZO",    "ionic_conductivity":1.62, "stability_window":5.20, "glass_transition_temp":27, "stability_score":0.88},
        {"material_id":"M9",  "sequence":"PEO-LiFSI-SN-LAGP",     "ionic_conductivity":1.48, "stability_window":4.95, "glass_transition_temp":32, "stability_score":0.85},
        {"material_id":"M10", "sequence":"PEGDA-LiTFSI-EC-Al2O3", "ionic_conductivity":1.78, "stability_window":5.15, "glass_transition_temp":30, "stability_score":0.91},
        {"material_id":"M11", "sequence":"PEO-LiTFSI-DMC-SiO2",   "ionic_conductivity":1.05, "stability_window":4.55, "glass_transition_temp":40, "stability_score":0.77},
        {"material_id":"M12", "sequence":"PVDF-LiFSI-SN-ZrO2",    "ionic_conductivity":1.50, "stability_window":5.05, "glass_transition_temp":33, "stability_score":0.86},
        {"material_id":"M13", "sequence":"PEG-LiTFSI-PC-Al2O3",   "ionic_conductivity":1.35, "stability_window":4.75, "glass_transition_temp":35, "stability_score":0.81},
        {"material_id":"M14", "sequence":"PEO-LiFSI-EC-LLZO",     "ionic_conductivity":1.88, "stability_window":5.25, "glass_transition_temp":26, "stability_score":0.92},
        {"material_id":"M15", "sequence":"PCL-LiBOB-SN-TiO2",     "ionic_conductivity":1.12, "stability_window":4.65, "glass_transition_temp":39, "stability_score":0.79},
        {"material_id":"M16", "sequence":"PEO-LiTFSI-SN-LATP",    "ionic_conductivity":1.57, "stability_window":5.00, "glass_transition_temp":31, "stability_score":0.87},
        {"material_id":"M17", "sequence":"PEGDA-LiFSI-PC-ZrO2",   "ionic_conductivity":1.66, "stability_window":5.08, "glass_transition_temp":30, "stability_score":0.88},
        {"material_id":"M18", "sequence":"PVDF-LiTFSI-EC-SiO2",   "ionic_conductivity":1.22, "stability_window":4.70, "glass_transition_temp":37, "stability_score":0.80},
        {"material_id":"M19", "sequence":"PEO-LiFSI-SN-Al2O3",    "ionic_conductivity":1.76, "stability_window":5.18, "glass_transition_temp":27, "stability_score":0.90},
        {"material_id":"M20", "sequence":"PEG-LiTFSI-EC-LLZO",    "ionic_conductivity":1.69, "stability_window":5.12, "glass_transition_temp":29, "stability_score":0.89},
        {"material_id":"M21", "sequence":"PEO-LiDFOB-PC-LATP",    "ionic_conductivity":1.08, "stability_window":4.50, "glass_transition_temp":41, "stability_score":0.76},
        {"material_id":"M22", "sequence":"PCL-LiFSI-SN-LLZO",     "ionic_conductivity":1.63, "stability_window":5.16, "glass_transition_temp":28, "stability_score":0.88},
        {"material_id":"M23", "sequence":"PEGDA-LiTFSI-SN-ZrO2",  "ionic_conductivity":1.74, "stability_window":5.14, "glass_transition_temp":29, "stability_score":0.90},
        {"material_id":"M24", "sequence":"PEO-LiFSI-PC-Al2O3",    "ionic_conductivity":1.28, "stability_window":4.72, "glass_transition_temp":36, "stability_score":0.82},
        {"material_id":"M25", "sequence":"PVDF-LiBOB-EC-TiO2",    "ionic_conductivity":0.98, "stability_window":4.42, "glass_transition_temp":43, "stability_score":0.75},
        {"material_id":"M26", "sequence":"PEGDA-LiDFOB-SN-LLZO",  "ionic_conductivity":1.71, "stability_window":5.09, "glass_transition_temp":28, "stability_score":0.89},
        {"material_id":"M27", "sequence":"PEO-LiFSI-DME-LAGP",    "ionic_conductivity":1.40, "stability_window":4.88, "glass_transition_temp":33, "stability_score":0.84},
        {"material_id":"M28", "sequence":"PVDF-LiFSI-EC-LLZO",    "ionic_conductivity":1.59, "stability_window":5.11, "glass_transition_temp":30, "stability_score":0.87},
        {"material_id":"M29", "sequence":"PEG-LiBOB-SN-ZrO2",     "ionic_conductivity":1.18, "stability_window":4.68, "glass_transition_temp":38, "stability_score":0.80},
        {"material_id":"M30", "sequence":"PEO-LiTFSI-EC-LLZO",    "ionic_conductivity":1.81, "stability_window":5.20, "glass_transition_temp":27, "stability_score":0.91},
    ]
    return pd.DataFrame(rows)


# ============================================================
# Helpers
# ============================================================
def ensure_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    df = df.copy()
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["sequence"] = df["sequence"].fillna("").astype(str)
    df = df.dropna(subset=cols)
    df = df[df["sequence"].str.strip() != ""].reset_index(drop=True)
    return df


def tokenize_sequence(seq: str) -> List[str]:
    if not isinstance(seq, str) or not seq.strip():
        return ["<UNK>"]
    toks = [x.strip() for x in seq.split("-") if x.strip()]
    return toks if toks else ["<UNK>"]


def build_vocab(sequences: List[str]) -> Dict[str, int]:
    vocab = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for seq in sequences:
        for tok in tokenize_sequence(seq):
            if tok not in vocab:
                vocab[tok] = len(vocab)
    return vocab


def encode_sequence(seq: str, vocab: Dict[str, int], max_len: int = MAX_LEN) -> List[int]:
    toks = ["<SOS>"] + tokenize_sequence(seq) + ["<EOS>"]
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in toks]

    if len(ids) < max_len:
        ids += [vocab["<PAD>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
        if ids[-1] not in (vocab["<EOS>"], vocab["<PAD>"]):
            ids[-1] = vocab["<EOS>"]
    return ids


def decode_ids(ids: List[int], inv_vocab: Dict[int, str]) -> str:
    toks = []
    for idx in ids:
        tok = inv_vocab.get(int(idx), "<UNK>")
        if tok in ("<PAD>", "<SOS>"):
            continue
        if tok == "<EOS>":
            break
        toks.append(tok)
    return "-".join(toks)


def sequence_to_flat_onehot(seq_ids: List[int], vocab_size: int) -> np.ndarray:
    arr = np.zeros((len(seq_ids), vocab_size), dtype=np.float32)
    for i, idx in enumerate(seq_ids):
        if 0 <= idx < vocab_size:
            arr[i, idx] = 1.0
    return arr.reshape(-1)


def rmse_distance(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.sqrt(np.mean((a - b) ** 2)))


def save_json(obj: Dict[str, Any], path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def safe_float(x, default=0.0):
    try:
        return float(x)
    except Exception:
        return default


# ============================================================
# Scaler
# ============================================================
class PropertyScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, x: np.ndarray):
        self.mean_ = x.mean(axis=0)
        self.std_ = x.std(axis=0) + 1e-8
        return self

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean_) / self.std_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * self.std_ + self.mean_

    def to_dict(self):
        return {"mean": self.mean_.tolist(), "std": self.std_.tolist()}

    @classmethod
    def from_dict(cls, d):
        obj = cls()
        obj.mean_ = np.array(d["mean"], dtype=np.float32)
        obj.std_ = np.array(d["std"], dtype=np.float32)
        return obj


# ============================================================
# Dataset
# ============================================================
class SequenceDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: Dict[str, int], scaler: PropertyScaler):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.scaler = scaler

        self.seq_ids = [encode_sequence(s, vocab, MAX_LEN) for s in self.df["sequence"].tolist()]
        props = self.df[TARGET_COLUMNS].values.astype(np.float32)
        self.props_scaled = scaler.transform(props).astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.seq_ids[idx], dtype=torch.long),
            torch.tensor(self.props_scaled[idx], dtype=torch.float32),
        )


# ============================================================
# Models
# ============================================================
class ConditionalVAE(nn.Module):
    def __init__(self, vocab_size: int, prop_dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)

        self.encoder = nn.Sequential(
            nn.Linear(MAX_LEN * EMBED_DIM + prop_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
        )
        self.mu_layer = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.logvar_layer = nn.Linear(HIDDEN_DIM, LATENT_DIM)

        self.decoder = nn.Sequential(
            nn.Linear(LATENT_DIM + prop_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, MAX_LEN * vocab_size),
        )

    def encode(self, seq_ids: torch.Tensor, props: torch.Tensor):
        emb = self.embedding(seq_ids)
        emb_flat = emb.reshape(seq_ids.size(0), -1)
        x = torch.cat([emb_flat, props], dim=1)
        h = self.encoder(x)
        mu = self.mu_layer(h)
        logvar = self.logvar_layer(h)
        return mu, logvar

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z: torch.Tensor, props: torch.Tensor):
        x = torch.cat([z, props], dim=1)
        logits = self.decoder(x)
        logits = logits.view(-1, MAX_LEN, self.vocab_size)
        return logits

    def forward(self, seq_ids: torch.Tensor, props: torch.Tensor):
        mu, logvar = self.encode(seq_ids, props)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z, props)
        return logits, mu, logvar

    def generate(self, props: torch.Tensor, n: int):
        z = torch.randn(n, LATENT_DIM, device=props.device)
        logits = self.decode(z, props)
        pred_ids = torch.argmax(logits, dim=-1)
        return pred_ids


def cvae_loss_fn(logits, targets, mu, logvar, pad_idx=0):
    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=pad_idx,
    )
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    total = recon + 0.01 * kl
    return total


class SurrogateMLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, output_dim),
        )

    def forward(self, x):
        return self.net(x)


# ============================================================
# Global state
# ============================================================
app = Flask(__name__)

GLOBAL_DF = None
GLOBAL_VOCAB = None
GLOBAL_INV_VOCAB = None
GLOBAL_SCALER = None
GLOBAL_CVAE = None
GLOBAL_SURROGATE = None


# ============================================================
# Training utilities
# ============================================================
def build_surrogate_xy(df: pd.DataFrame, vocab: Dict[str, int]) -> Tuple[np.ndarray, np.ndarray]:
    X, y = [], []
    vocab_size = len(vocab)

    for _, row in df.iterrows():
        seq_ids = encode_sequence(row["sequence"], vocab, MAX_LEN)
        X.append(sequence_to_flat_onehot(seq_ids, vocab_size))
        y.append(row[TARGET_COLUMNS].values.astype(np.float32))

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def save_artifacts(vocab, scaler, cvae, surrogate):
    save_json(vocab, os.path.join(ARTIFACT_DIR, "vocab.json"))
    save_json(scaler.to_dict(), os.path.join(ARTIFACT_DIR, "scaler.json"))
    torch.save(cvae.state_dict(), os.path.join(ARTIFACT_DIR, "cvae.pt"))
    torch.save(surrogate.state_dict(), os.path.join(ARTIFACT_DIR, "surrogate.pt"))


def load_artifacts():
    vocab_path = os.path.join(ARTIFACT_DIR, "vocab.json")
    scaler_path = os.path.join(ARTIFACT_DIR, "scaler.json")
    cvae_path = os.path.join(ARTIFACT_DIR, "cvae.pt")
    surrogate_path = os.path.join(ARTIFACT_DIR, "surrogate.pt")

    if not all(os.path.exists(p) for p in [vocab_path, scaler_path, cvae_path, surrogate_path]):
        return None, None, None, None, None

    vocab = load_json(vocab_path)
    inv_vocab = {v: k for k, v in vocab.items()}
    scaler = PropertyScaler.from_dict(load_json(scaler_path))

    cvae = ConditionalVAE(vocab_size=len(vocab), prop_dim=len(TARGET_COLUMNS)).to(DEVICE)
    surrogate = SurrogateMLP(
        input_dim=MAX_LEN * len(vocab),
        output_dim=len(TARGET_COLUMNS),
    ).to(DEVICE)

    cvae.load_state_dict(torch.load(cvae_path, map_location=DEVICE))
    surrogate.load_state_dict(torch.load(surrogate_path, map_location=DEVICE))
    cvae.eval()
    surrogate.eval()

    return vocab, inv_vocab, scaler, cvae, surrogate


def train_models():
    global GLOBAL_DF, GLOBAL_VOCAB, GLOBAL_INV_VOCAB, GLOBAL_SCALER, GLOBAL_CVAE, GLOBAL_SURROGATE

    df = build_internal_dataset()
    df = ensure_numeric(df, TARGET_COLUMNS)

    vocab = build_vocab(df["sequence"].tolist())
    inv_vocab = {v: k for k, v in vocab.items()}
    scaler = PropertyScaler().fit(df[TARGET_COLUMNS].values.astype(np.float32))

    # CVAE
    dataset = SequenceDataset(df, vocab, scaler)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    cvae = ConditionalVAE(vocab_size=len(vocab), prop_dim=len(TARGET_COLUMNS)).to(DEVICE)
    cvae_opt = torch.optim.Adam(cvae.parameters(), lr=LR)

    cvae_losses = []
    for _ in range(EPOCHS_CVAE):
        cvae.train()
        epoch_loss = 0.0
        for seq_ids, props in loader:
            seq_ids = seq_ids.to(DEVICE)
            props = props.to(DEVICE)

            logits, mu, logvar = cvae(seq_ids, props)
            loss = cvae_loss_fn(logits, seq_ids, mu, logvar, pad_idx=vocab["<PAD>"])

            cvae_opt.zero_grad()
            loss.backward()
            cvae_opt.step()

            epoch_loss += loss.item()

        cvae_losses.append(epoch_loss / max(1, len(loader)))

    # Surrogate
    X, y = build_surrogate_xy(df, vocab)
    y_scaled = scaler.transform(y)

    X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    y_t = torch.tensor(y_scaled, dtype=torch.float32).to(DEVICE)

    surrogate = SurrogateMLP(input_dim=X.shape[1], output_dim=y.shape[1]).to(DEVICE)
    s_opt = torch.optim.Adam(surrogate.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    surrogate_losses = []
    for _ in range(EPOCHS_SURROGATE):
        surrogate.train()
        preds = surrogate(X_t)
        loss = loss_fn(preds, y_t)

        s_opt.zero_grad()
        loss.backward()
        s_opt.step()

        surrogate_losses.append(float(loss.item()))

    cvae.eval()
    surrogate.eval()

    save_artifacts(vocab, scaler, cvae, surrogate)

    GLOBAL_DF = df
    GLOBAL_VOCAB = vocab
    GLOBAL_INV_VOCAB = inv_vocab
    GLOBAL_SCALER = scaler
    GLOBAL_CVAE = cvae
    GLOBAL_SURROGATE = surrogate

    return {
        "status": "trained",
        "rows_used": int(len(df)),
        "vocab_size": int(len(vocab)),
        "cvae_final_loss": float(cvae_losses[-1]),
        "surrogate_final_loss": float(surrogate_losses[-1]),
    }


def models_ready() -> bool:
    return all([
        GLOBAL_DF is not None,
        GLOBAL_VOCAB is not None,
        GLOBAL_INV_VOCAB is not None,
        GLOBAL_SCALER is not None,
        GLOBAL_CVAE is not None,
        GLOBAL_SURROGATE is not None,
    ])


# ============================================================
# Prediction / explanation
# ============================================================
def predict_sequence_properties(sequence: str) -> Dict[str, float]:
    seq_ids = encode_sequence(sequence, GLOBAL_VOCAB, MAX_LEN)
    x = sequence_to_flat_onehot(seq_ids, len(GLOBAL_VOCAB)).reshape(1, -1)
    x_t = torch.tensor(x, dtype=torch.float32).to(DEVICE)

    GLOBAL_SURROGATE.eval()
    with torch.no_grad():
        pred_scaled = GLOBAL_SURROGATE(x_t).cpu().numpy()

    pred = GLOBAL_SCALER.inverse_transform(pred_scaled)[0]
    return {TARGET_COLUMNS[i]: float(pred[i]) for i in range(len(TARGET_COLUMNS))}


def predict_with_uncertainty(sequence: str, runs: int = MC_DROPOUT_RUNS) -> Dict[str, Dict[str, float]]:
    seq_ids = encode_sequence(sequence, GLOBAL_VOCAB, MAX_LEN)
    x = sequence_to_flat_onehot(seq_ids, len(GLOBAL_VOCAB)).reshape(1, -1)
    x_t = torch.tensor(x, dtype=torch.float32).to(DEVICE)

    preds = []
    GLOBAL_SURROGATE.train()
    for _ in range(runs):
        with torch.no_grad():
            pred_scaled = GLOBAL_SURROGATE(x_t).cpu().numpy()
            preds.append(GLOBAL_SCALER.inverse_transform(pred_scaled)[0])
    GLOBAL_SURROGATE.eval()

    preds = np.array(preds, dtype=np.float32)
    mean_pred = preds.mean(axis=0)
    std_pred = preds.std(axis=0)

    return {
        "mean": {TARGET_COLUMNS[i]: float(mean_pred[i]) for i in range(len(TARGET_COLUMNS))},
        "std": {TARGET_COLUMNS[i]: float(std_pred[i]) for i in range(len(TARGET_COLUMNS))},
    }


def simple_token_importance(sequence: str) -> Dict[str, float]:
    tokens = tokenize_sequence(sequence)
    if len(tokens) == 1:
        return {tokens[0]: 1.0}

    base = predict_sequence_properties(sequence)
    base_arr = np.array([base[c] for c in TARGET_COLUMNS], dtype=np.float32)

    impacts = {}
    for i, tok in enumerate(tokens):
        reduced = tokens[:i] + tokens[i+1:]
        reduced_seq = "-".join(reduced) if reduced else "<UNK>"
        pred = predict_sequence_properties(reduced_seq)
        pred_arr = np.array([pred[c] for c in TARGET_COLUMNS], dtype=np.float32)
        impacts[tok] = rmse_distance(base_arr, pred_arr)

    total = sum(impacts.values()) + 1e-8
    return {k: float(v / total) for k, v in impacts.items()}


# ============================================================
# Inverse design
# ============================================================
def generate_candidates(target: Dict[str, float], num_samples: int = 100, top_k: int = 10):
    target_arr = np.array([float(target[c]) for c in TARGET_COLUMNS], dtype=np.float32).reshape(1, -1)
    target_scaled = GLOBAL_SCALER.transform(target_arr)

    props_tensor = torch.tensor(
        np.repeat(target_scaled, num_samples, axis=0),
        dtype=torch.float32,
        device=DEVICE
    )

    GLOBAL_CVAE.eval()
    with torch.no_grad():
        batch_ids = GLOBAL_CVAE.generate(props_tensor, num_samples).cpu().numpy()

    candidates = []
    seen = set()

    for ids in batch_ids:
        seq = decode_ids(ids.tolist(), GLOBAL_INV_VOCAB)
        if not seq or seq in seen:
            continue
        seen.add(seq)

        pred = predict_sequence_properties(seq)
        pred_arr = np.array([pred[c] for c in TARGET_COLUMNS], dtype=np.float32)
        dist = rmse_distance(pred_arr, target_arr[0])
        unc = predict_with_uncertainty(seq, runs=10)["std"]

        candidates.append({
            "sequence": seq,
            "predicted_properties": pred,
            "uncertainty_std": unc,
            "distance_to_target": float(dist),
        })

    candidates = sorted(candidates, key=lambda x: x["distance_to_target"])
    return candidates[:top_k]


# ============================================================
# Plot helpers
# ============================================================
def fig_to_base64() -> str:
    buf = io.BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", bbox_inches="tight")
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


def plot_dataset_summary(df: pd.DataFrame) -> str:
    means = [df[c].mean() for c in TARGET_COLUMNS]
    plt.figure(figsize=(8, 4))
    plt.bar(TARGET_COLUMNS, means)
    plt.xticks(rotation=20)
    plt.ylabel("Mean value")
    plt.title("Built-in Training Dataset Mean Property Values")
    return fig_to_base64()


def plot_token_importance(importance: Dict[str, float]) -> str:
    plt.figure(figsize=(8, 4))
    plt.bar(list(importance.keys()), list(importance.values()))
    plt.xticks(rotation=20)
    plt.ylabel("Relative importance")
    plt.title("Token Importance")
    return fig_to_base64()


# ============================================================
# HTML
# ============================================================
INDEX_HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Inverse Materials Design</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body { font-family: Arial, sans-serif; background: #f4f7fb; margin: 0; padding: 0; color: #222; }
        .container { max-width: 1100px; margin: 30px auto; background: white; border-radius: 18px; padding: 24px; box-shadow: 0 8px 30px rgba(0,0,0,0.08); }
        .grid { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; }
        .card { background: #fafcff; border: 1px solid #dde7f5; border-radius: 14px; padding: 16px; }
        input, button { width: 100%; padding: 10px; margin-top: 8px; margin-bottom: 10px; border-radius: 10px; border: 1px solid #cad5e2; box-sizing: border-box; font-size: 14px; }
        button { background: #1459c7; color: white; border: none; cursor: pointer; font-weight: bold; }
        button:hover { background: #0f49a5; }
        .mono { font-family: Consolas, monospace; background: #f3f6fa; padding: 10px; border-radius: 8px; overflow-x: auto; white-space: pre-wrap; }
        img { max-width: 100%; border-radius: 12px; margin-top: 12px; }
        table { width: 100%; border-collapse: collapse; margin-top: 12px; }
        th, td { border: 1px solid #dce5ef; padding: 8px; text-align: left; font-size: 14px; vertical-align: top; }
        th { background: #eef4fb; }
        .status { padding: 10px; border-radius: 10px; background: #eef7ea; border: 1px solid #cfe8c6; margin-bottom: 16px; }
        @media (max-width: 900px) { .grid { grid-template-columns: 1fr; } }
    </style>
</head>
<body>
<div class="container">
    <h1>Generative AI for Inverse Materials Design</h1>
    <div class="status">
        <b>Device:</b> {{ device }} |
        <b>Models Ready:</b> {{ ready }} |
        <b>Built-in Rows:</b> {{ rows }}
    </div>

    <div class="grid">
        <div class="card">
            <h2>1. Train Built-in Model</h2>
            <form method="post" action="/web/train">
                <button type="submit">Train Using Internal Dataset</button>
            </form>
            {% if train_result %}
                <div class="mono">{{ train_result }}</div>
            {% endif %}
            {% if dataset_plot %}
                <img src="data:image/png;base64,{{ dataset_plot }}">
            {% endif %}
        </div>

        <div class="card">
            <h2>2. Predict One Sequence</h2>
            <form method="post" action="/web/predict">
                <label>Sequence</label>
                <input type="text" name="sequence" value="{{ sequence_value or 'PEO-LiTFSI-SN-Al2O3' }}">
                <button type="submit">Predict</button>
            </form>
            {% if predict_result %}
                <div class="mono">{{ predict_result }}</div>
            {% endif %}
            {% if imp_plot %}
                <img src="data:image/png;base64,{{ imp_plot }}">
            {% endif %}
        </div>
    </div>

    <div class="card" style="margin-top:18px;">
        <h2>3. Generate Candidates from Desired Properties</h2>
        <form method="post" action="/web/generate">
            <div class="grid">
                <div><label>Ionic Conductivity</label><input type="text" name="ionic_conductivity" value="{{ gen_form.get('ionic_conductivity', '1.70') }}"></div>
                <div><label>Stability Window</label><input type="text" name="stability_window" value="{{ gen_form.get('stability_window', '5.10') }}"></div>
                <div><label>Glass Transition Temp</label><input type="text" name="glass_transition_temp" value="{{ gen_form.get('glass_transition_temp', '29') }}"></div>
                <div><label>Stability Score</label><input type="text" name="stability_score" value="{{ gen_form.get('stability_score', '0.90') }}"></div>
                <div><label>Number of Samples</label><input type="text" name="num_samples" value="{{ gen_form.get('num_samples', '100') }}"></div>
                <div><label>Top K</label><input type="text" name="top_k" value="{{ gen_form.get('top_k', '10') }}"></div>
            </div>
            <button type="submit">Generate</button>
        </form>

        {% if generated_candidates %}
        <table>
            <thead>
                <tr>
                    <th>#</th>
                    <th>Sequence</th>
                    <th>Distance</th>
                    <th>Predicted Properties</th>
                    <th>Uncertainty Std</th>
                </tr>
            </thead>
            <tbody>
                {% for row in generated_candidates %}
                <tr>
                    <td>{{ loop.index }}</td>
                    <td>{{ row["sequence"] }}</td>
                    <td>{{ "%.4f"|format(row["distance_to_target"]) }}</td>
                    <td><div class="mono">{{ row["predicted_properties"] }}</div></td>
                    <td><div class="mono">{{ row["uncertainty_std"] }}</div></td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
        {% endif %}
    </div>

    <div class="card" style="margin-top:18px;">
        <h2>API Endpoints</h2>
        <div class="mono">GET /health
POST /train
POST /predict
POST /generate</div>
    </div>

    {% if error %}
    <div class="card" style="margin-top:18px; border:1px solid #f1b7b7; background:#fff3f3;">
        <h2>Error</h2>
        <div class="mono">{{ error }}</div>
    </div>
    {% endif %}
</div>
</body>
</html>
"""


# ============================================================
# Render helper
# ============================================================
def render_home(**kwargs):
    rows = len(GLOBAL_DF) if GLOBAL_DF is not None else len(build_internal_dataset())
    return render_template_string(
        INDEX_HTML,
        device=DEVICE,
        ready=models_ready(),
        rows=rows,
        train_result=kwargs.get("train_result"),
        dataset_plot=kwargs.get("dataset_plot"),
        predict_result=kwargs.get("predict_result"),
        imp_plot=kwargs.get("imp_plot"),
        generated_candidates=kwargs.get("generated_candidates"),
        sequence_value=kwargs.get("sequence_value"),
        gen_form=kwargs.get("gen_form", {}),
        error=kwargs.get("error"),
    )


# ============================================================
# API routes
# ============================================================
@app.route("/", methods=["GET"])
def home():
    return render_home()


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "device": DEVICE,
        "models_ready": models_ready(),
        "built_in_rows": int(len(build_internal_dataset())),
    })


@app.route("/train", methods=["POST"])
def api_train():
    try:
        result = train_models()
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/predict", methods=["POST"])
def api_predict():
    try:
        if not models_ready():
            return jsonify({"error": "Models are not ready. Call /train first."}), 400

        payload = request.get_json(silent=True) or {}
        sequence = str(payload.get("sequence", "")).strip()
        if not sequence:
            return jsonify({"error": "Missing sequence"}), 400

        pred = predict_sequence_properties(sequence)
        unc = predict_with_uncertainty(sequence)
        imp = simple_token_importance(sequence)

        return jsonify({
            "sequence": sequence,
            "predicted_properties": pred,
            "uncertainty_std": unc["std"],
            "token_importance": imp,
        })
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/generate", methods=["POST"])
def api_generate():
    try:
        if not models_ready():
            return jsonify({"error": "Models are not ready. Call /train first."}), 400

        payload = request.get_json(silent=True) or {}
        target = payload.get("target", {})
        num_samples = int(payload.get("num_samples", 100))
        top_k = int(payload.get("top_k", 10))

        candidates = generate_candidates(target, num_samples=num_samples, top_k=top_k)

        return jsonify({
            "target": target,
            "num_returned": len(candidates),
            "candidates": candidates
        })
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


# ============================================================
# Web routes
# ============================================================
@app.route("/web/train", methods=["POST"])
def web_train():
    try:
        result = train_models()
        dataset_plot = plot_dataset_summary(GLOBAL_DF)
        return render_home(
            train_result=json.dumps(result, indent=2),
            dataset_plot=dataset_plot,
        )
    except Exception as e:
        return render_home(error=f"{str(e)}\n\n{traceback.format_exc()}")


@app.route("/web/predict", methods=["POST"])
def web_predict():
    try:
        if not models_ready():
            return render_home(error="Models are not ready. Click Train first.")

        sequence = request.form.get("sequence", "").strip()
        if not sequence:
            return render_home(error="Please enter a sequence.")

        pred = predict_sequence_properties(sequence)
        unc = predict_with_uncertainty(sequence)["std"]
        imp = simple_token_importance(sequence)

        result = {
            "sequence": sequence,
            "predicted_properties": pred,
            "uncertainty_std": unc,
            "token_importance": imp,
        }

        return render_home(
            predict_result=json.dumps(result, indent=2),
            imp_plot=plot_token_importance(imp),
            sequence_value=sequence,
        )
    except Exception as e:
        return render_home(error=f"{str(e)}\n\n{traceback.format_exc()}")


@app.route("/web/generate", methods=["POST"])
def web_generate():
    try:
        if not models_ready():
            return render_home(error="Models are not ready. Click Train first.")

        gen_form = {
            "ionic_conductivity": request.form.get("ionic_conductivity", "1.70"),
            "stability_window": request.form.get("stability_window", "5.10"),
            "glass_transition_temp": request.form.get("glass_transition_temp", "29"),
            "stability_score": request.form.get("stability_score", "0.90"),
            "num_samples": request.form.get("num_samples", "100"),
            "top_k": request.form.get("top_k", "10"),
        }

        target = {
            "ionic_conductivity": safe_float(gen_form["ionic_conductivity"]),
            "stability_window": safe_float(gen_form["stability_window"]),
            "glass_transition_temp": safe_float(gen_form["glass_transition_temp"]),
            "stability_score": safe_float(gen_form["stability_score"]),
        }

        num_samples = int(float(gen_form["num_samples"]))
        top_k = int(float(gen_form["top_k"]))

        candidates = generate_candidates(target, num_samples=num_samples, top_k=top_k)

        return render_home(
            generated_candidates=candidates,
            gen_form=gen_form,
        )
    except Exception as e:
        return render_home(error=f"{str(e)}\n\n{traceback.format_exc()}")


# ============================================================
# Startup
# ============================================================
def try_load_existing():
    global GLOBAL_DF, GLOBAL_VOCAB, GLOBAL_INV_VOCAB, GLOBAL_SCALER, GLOBAL_CVAE, GLOBAL_SURROGATE

    GLOBAL_DF = build_internal_dataset()

    loaded = load_artifacts()
    if loaded[0] is not None:
        GLOBAL_VOCAB, GLOBAL_INV_VOCAB, GLOBAL_SCALER, GLOBAL_CVAE, GLOBAL_SURROGATE = loaded


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    try_load_existing()
    app.run(host="127.0.0.1", port=5024, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5024
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:53:11] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:53:11] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:53:17] "POST /web/train HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:53:23] "POST /web/predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 11:53:29] "POST /web/generate HTTP/1.1" 200 -


In [ ]:
#!/usr/bin/env python3
"""
GenMatDesign-AI
========================================================
Generative AI for Inverse Materials Design
Premium GitHub-ready Flask dashboard + API

Focus:
- Solid-state battery electrolytes

Capabilities:
- Conditional VAE candidate generation
- Surrogate property prediction
- Inverse design from desired properties
- Uncertainty estimation via MC dropout
- Token-level explanation
- Dashboard UI
- CSV export of generated candidates
- REST API

Author: Your Name
Version: 2.0.0
========================================================
"""

import io
import json
import base64
import random
import traceback
from typing import Dict, List, Any

import numpy as np
import pandas as pd
from flask import Flask, request, jsonify, render_template_string, make_response

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# Configuration
# =========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TARGET_COLUMNS = [
    "ionic_conductivity",
    "stability_window",
    "glass_transition_temp",
    "stability_score",
]

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]

MAX_LEN = 14
EMBED_DIM = 64
HIDDEN_DIM = 128
LATENT_DIM = 32
EPOCHS_CVAE = 60
EPOCHS_SURROGATE = 120
LR = 1e-3
MC_RUNS = 20
NUM_GENERATION_SAMPLES = 120


# =========================================================
# Flask app
# =========================================================
app = Flask(__name__)

GLOBAL = {
    "df": None,
    "vocab": None,
    "inv_vocab": None,
    "scaler": None,
    "cvae": None,
    "surrogate": None,
    "last_generated": None,
}


# =========================================================
# Built-in electrolyte library
# =========================================================
def load_internal_data() -> pd.DataFrame:
    rows = [
        ("PEO-LiTFSI-SN-Al2O3", 1.20, 4.80, 38, 0.82),
        ("PEGDA-LiFSI-EC-LLZO", 1.85, 5.10, 29, 0.90),
        ("PEO-LiFSI-PC-TiO2", 1.10, 4.60, 42, 0.78),
        ("PVDF-LiTFSI-SN-LATP", 1.55, 5.30, 31, 0.87),
        ("PEO-LiTFSI-EC-ZrO2", 1.30, 4.90, 36, 0.83),
        ("PEG-LiFSI-SN-Al2O3", 1.72, 5.00, 28, 0.89),
        ("PEO-LiBOB-PC-SiO2", 0.95, 4.40, 44, 0.74),
        ("PCL-LiTFSI-SN-LLZO", 1.62, 5.20, 27, 0.88),
        ("PEO-LiFSI-SN-LAGP", 1.48, 4.95, 32, 0.85),
        ("PEGDA-LiTFSI-EC-Al2O3", 1.78, 5.15, 30, 0.91),
        ("PEO-LiTFSI-DMC-SiO2", 1.05, 4.55, 40, 0.77),
        ("PVDF-LiFSI-SN-ZrO2", 1.50, 5.05, 33, 0.86),
        ("PEG-LiTFSI-PC-Al2O3", 1.35, 4.75, 35, 0.81),
        ("PEO-LiFSI-EC-LLZO", 1.88, 5.25, 26, 0.92),
        ("PCL-LiBOB-SN-TiO2", 1.12, 4.65, 39, 0.79),
        ("PEO-LiTFSI-SN-LATP", 1.57, 5.00, 31, 0.87),
        ("PEGDA-LiFSI-PC-ZrO2", 1.66, 5.08, 30, 0.88),
        ("PVDF-LiTFSI-EC-SiO2", 1.22, 4.70, 37, 0.80),
        ("PEO-LiFSI-SN-Al2O3", 1.76, 5.18, 27, 0.90),
        ("PEG-LiTFSI-EC-LLZO", 1.69, 5.12, 29, 0.89),
        ("PEO-LiDFOB-PC-LATP", 1.08, 4.50, 41, 0.76),
        ("PCL-LiFSI-SN-LLZO", 1.63, 5.16, 28, 0.88),
        ("PEGDA-LiTFSI-SN-ZrO2", 1.74, 5.14, 29, 0.90),
        ("PEO-LiFSI-PC-Al2O3", 1.28, 4.72, 36, 0.82),
        ("PVDF-LiBOB-EC-TiO2", 0.98, 4.42, 43, 0.75),
        ("PEGDA-LiDFOB-SN-LLZO", 1.71, 5.09, 28, 0.89),
        ("PEO-LiFSI-DME-LAGP", 1.40, 4.88, 33, 0.84),
        ("PVDF-LiFSI-EC-LLZO", 1.59, 5.11, 30, 0.87),
        ("PEG-LiBOB-SN-ZrO2", 1.18, 4.68, 38, 0.80),
        ("PEO-LiTFSI-EC-LLZO", 1.81, 5.20, 27, 0.91),
        ("PEGDA-LiTFSI-SN-LLZO", 1.84, 5.22, 28, 0.91),
        ("PEO-LiFSI-SN-ZrO2", 1.61, 5.00, 31, 0.86),
        ("PVDF-LiDFOB-SN-Al2O3", 1.36, 4.89, 34, 0.83),
        ("PCL-LiFSI-EC-LATP", 1.46, 4.97, 32, 0.85),
        ("PEG-LiFSI-EC-ZrO2", 1.58, 5.03, 31, 0.87),
    ]
    return pd.DataFrame(rows, columns=["sequence"] + TARGET_COLUMNS)


# =========================================================
# Tokenization and vocabulary
# =========================================================
def tokenize(seq: str) -> List[str]:
    if not isinstance(seq, str) or not seq.strip():
        return ["<UNK>"]
    return [tok.strip() for tok in seq.split("-") if tok.strip()]


def build_vocab(sequences: List[str]) -> Dict[str, int]:
    vocab = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    for seq in sequences:
        for tok in tokenize(seq):
            if tok not in vocab:
                vocab[tok] = len(vocab)
    return vocab


def encode_sequence(seq: str, vocab: Dict[str, int]) -> List[int]:
    tokens = ["<SOS>"] + tokenize(seq) + ["<EOS>"]
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokens]
    if len(ids) < MAX_LEN:
        ids += [vocab["<PAD>"]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]
        if ids[-1] not in (vocab["<EOS>"], vocab["<PAD>"]):
            ids[-1] = vocab["<EOS>"]
    return ids


def decode_ids(ids: List[int], inv_vocab: Dict[int, str]) -> str:
    out = []
    for idx in ids:
        tok = inv_vocab.get(int(idx), "<UNK>")
        if tok in ("<PAD>", "<SOS>"):
            continue
        if tok == "<EOS>":
            break
        out.append(tok)
    return "-".join(out)


def onehot_flat(ids: List[int], vocab_size: int) -> np.ndarray:
    arr = np.zeros((len(ids), vocab_size), dtype=np.float32)
    for i, idx in enumerate(ids):
        if 0 <= idx < vocab_size:
            arr[i, idx] = 1.0
    return arr.reshape(-1)


# =========================================================
# Property scaler
# =========================================================
class PropertyScaler:
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, x: np.ndarray):
        self.mean_ = x.mean(axis=0)
        self.std_ = x.std(axis=0) + 1e-8
        return self

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean_) / self.std_

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        return x * self.std_ + self.mean_


# =========================================================
# Models
# =========================================================
class CVAE(nn.Module):
    def __init__(self, vocab_size: int, prop_dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)

        self.encoder = nn.Sequential(
            nn.Linear(MAX_LEN * EMBED_DIM + prop_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
        )
        self.mu_layer = nn.Linear(HIDDEN_DIM, LATENT_DIM)
        self.logvar_layer = nn.Linear(HIDDEN_DIM, LATENT_DIM)

        self.decoder = nn.Sequential(
            nn.Linear(LATENT_DIM + prop_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, MAX_LEN * vocab_size),
        )

    def forward(self, x: torch.Tensor, p: torch.Tensor):
        e = self.emb(x).reshape(x.size(0), -1)
        h = self.encoder(torch.cat([e, p], dim=1))
        mu = self.mu_layer(h)
        logvar = self.logvar_layer(h)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        out = self.decoder(torch.cat([z, p], dim=1))
        out = out.view(-1, MAX_LEN, self.vocab_size)
        return out, mu, logvar

    def generate(self, p: torch.Tensor, n: int):
        z = torch.randn(n, LATENT_DIM, device=p.device)
        out = self.decoder(torch.cat([z, p], dim=1))
        out = out.view(-1, MAX_LEN, self.vocab_size)
        return out.argmax(dim=-1)


class Surrogate(nn.Module):
    def __init__(self, input_dim: int, output_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(64, output_dim),
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)


# =========================================================
# Training
# =========================================================
def train_models() -> Dict[str, Any]:
    df = load_internal_data()

    vocab = build_vocab(df["sequence"].tolist())
    inv_vocab = {v: k for k, v in vocab.items()}

    scaler = PropertyScaler().fit(df[TARGET_COLUMNS].values.astype(np.float32))

    # Train CVAE
    seq_ids = np.array([encode_sequence(s, vocab) for s in df["sequence"]], dtype=np.int64)
    props = scaler.transform(df[TARGET_COLUMNS].values.astype(np.float32))

    x_seq = torch.tensor(seq_ids, dtype=torch.long, device=DEVICE)
    x_prop = torch.tensor(props, dtype=torch.float32, device=DEVICE)

    cvae = CVAE(vocab_size=len(vocab), prop_dim=len(TARGET_COLUMNS)).to(DEVICE)
    opt = torch.optim.Adam(cvae.parameters(), lr=LR)

    cvae_losses = []
    for _ in range(EPOCHS_CVAE):
        cvae.train()
        logits, mu, logvar = cvae(x_seq, x_prop)
        recon = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            x_seq.reshape(-1),
            ignore_index=vocab["<PAD>"],
        )
        kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon + 0.01 * kl
        opt.zero_grad()
        loss.backward()
        opt.step()
        cvae_losses.append(float(loss.item()))

    # Train surrogate
    X = np.array([onehot_flat(encode_sequence(s, vocab), len(vocab)) for s in df["sequence"]], dtype=np.float32)
    y = props.astype(np.float32)

    x_train = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    y_train = torch.tensor(y, dtype=torch.float32, device=DEVICE)

    surrogate = Surrogate(input_dim=X.shape[1], output_dim=y.shape[1]).to(DEVICE)
    opt2 = torch.optim.Adam(surrogate.parameters(), lr=LR)

    surr_losses = []
    for _ in range(EPOCHS_SURROGATE):
        surrogate.train()
        pred = surrogate(x_train)
        loss = F.mse_loss(pred, y_train)
        opt2.zero_grad()
        loss.backward()
        opt2.step()
        surr_losses.append(float(loss.item()))

    GLOBAL.update({
        "df": df,
        "vocab": vocab,
        "inv_vocab": inv_vocab,
        "scaler": scaler,
        "cvae": cvae.eval(),
        "surrogate": surrogate.eval(),
        "last_generated": None,
    })

    return {
        "status": "trained",
        "rows_used": int(len(df)),
        "vocab_size": int(len(vocab)),
        "cvae_final_loss": float(cvae_losses[-1]),
        "surrogate_final_loss": float(surr_losses[-1]),
    }


def is_ready() -> bool:
    return all([
        GLOBAL["df"] is not None,
        GLOBAL["vocab"] is not None,
        GLOBAL["inv_vocab"] is not None,
        GLOBAL["scaler"] is not None,
        GLOBAL["cvae"] is not None,
        GLOBAL["surrogate"] is not None,
    ])


# =========================================================
# Inference
# =========================================================
def predict_sequence(sequence: str) -> Dict[str, float]:
    vocab = GLOBAL["vocab"]
    scaler = GLOBAL["scaler"]
    surrogate = GLOBAL["surrogate"]

    ids = encode_sequence(sequence, vocab)
    x = onehot_flat(ids, len(vocab)).reshape(1, -1)
    xt = torch.tensor(x, dtype=torch.float32, device=DEVICE)

    surrogate.eval()
    with torch.no_grad():
        pred_scaled = surrogate(xt).cpu().numpy()

    pred = scaler.inverse_transform(pred_scaled)[0]
    return {TARGET_COLUMNS[i]: float(pred[i]) for i in range(len(TARGET_COLUMNS))}


def predict_uncertainty(sequence: str, runs: int = MC_RUNS) -> Dict[str, Dict[str, float]]:
    vocab = GLOBAL["vocab"]
    scaler = GLOBAL["scaler"]
    surrogate = GLOBAL["surrogate"]

    ids = encode_sequence(sequence, vocab)
    x = onehot_flat(ids, len(vocab)).reshape(1, -1)
    xt = torch.tensor(x, dtype=torch.float32, device=DEVICE)

    preds = []
    surrogate.train()
    for _ in range(runs):
        with torch.no_grad():
            pred_scaled = surrogate(xt).cpu().numpy()
            pred = scaler.inverse_transform(pred_scaled)[0]
            preds.append(pred)
    surrogate.eval()

    preds = np.array(preds, dtype=np.float32)
    mean_pred = preds.mean(axis=0)
    std_pred = preds.std(axis=0)

    return {
        "mean": {TARGET_COLUMNS[i]: float(mean_pred[i]) for i in range(len(TARGET_COLUMNS))},
        "std": {TARGET_COLUMNS[i]: float(std_pred[i]) for i in range(len(TARGET_COLUMNS))},
    }


def token_importance(sequence: str) -> Dict[str, float]:
    toks = tokenize(sequence)
    if len(toks) == 1:
        return {toks[0]: 1.0}

    base = predict_sequence(sequence)
    base_arr = np.array([base[c] for c in TARGET_COLUMNS], dtype=np.float32)

    impacts = {}
    for i, tok in enumerate(toks):
        reduced = toks[:i] + toks[i+1:]
        reduced_seq = "-".join(reduced) if reduced else "<UNK>"
        pred = predict_sequence(reduced_seq)
        pred_arr = np.array([pred[c] for c in TARGET_COLUMNS], dtype=np.float32)
        impacts[tok] = float(np.sqrt(np.mean((base_arr - pred_arr) ** 2)))

    total = sum(impacts.values()) + 1e-8
    return {k: float(v / total) for k, v in impacts.items()}


def generate_candidates(target: Dict[str, float], top_k: int = 10) -> List[Dict[str, Any]]:
    scaler = GLOBAL["scaler"]
    cvae = GLOBAL["cvae"]
    inv_vocab = GLOBAL["inv_vocab"]

    target_vec = np.array([float(target[c]) for c in TARGET_COLUMNS], dtype=np.float32).reshape(1, -1)
    target_scaled = scaler.transform(target_vec)

    pt = torch.tensor(
        np.repeat(target_scaled, NUM_GENERATION_SAMPLES, axis=0),
        dtype=torch.float32,
        device=DEVICE,
    )

    cvae.eval()
    with torch.no_grad():
        ids_batch = cvae.generate(pt, NUM_GENERATION_SAMPLES).cpu().numpy()

    rows = []
    seen = set()

    for ids in ids_batch:
        seq = decode_ids(ids.tolist(), inv_vocab)
        if not seq or seq in seen:
            continue
        seen.add(seq)

        pred = predict_sequence(seq)
        pred_arr = np.array([pred[c] for c in TARGET_COLUMNS], dtype=np.float32)
        dist = float(np.sqrt(np.mean((pred_arr - target_vec[0]) ** 2)))
        unc = predict_uncertainty(seq, runs=10)["std"]

        rows.append({
            "sequence": seq,
            "predicted_properties": pred,
            "uncertainty_std": unc,
            "distance_to_target": dist,
        })

    rows = sorted(rows, key=lambda x: x["distance_to_target"])[:top_k]
    GLOBAL["last_generated"] = rows
    return rows


# =========================================================
# Plot helpers
# =========================================================
def fig_to_base64() -> str:
    buf = io.BytesIO()
    plt.tight_layout()
    plt.savefig(buf, format="png", bbox_inches="tight")
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


def plot_dataset_summary(df: pd.DataFrame) -> str:
    plt.figure(figsize=(8, 4))
    means = [df[c].mean() for c in TARGET_COLUMNS]
    plt.bar(TARGET_COLUMNS, means)
    plt.xticks(rotation=20)
    plt.ylabel("Mean")
    plt.title("Built-in Training Library: Mean Property Values")
    return fig_to_base64()


def plot_token_importance(imp: Dict[str, float]) -> str:
    plt.figure(figsize=(8, 4))
    plt.bar(list(imp.keys()), list(imp.values()))
    plt.xticks(rotation=25)
    plt.ylabel("Relative importance")
    plt.title("Token Importance Explanation")
    return fig_to_base64()


def plot_prediction(pred: Dict[str, float]) -> str:
    plt.figure(figsize=(8, 4))
    plt.bar(TARGET_COLUMNS, [pred[c] for c in TARGET_COLUMNS])
    plt.xticks(rotation=20)
    plt.ylabel("Predicted value")
    plt.title("Predicted Properties")
    return fig_to_base64()


# =========================================================
# UI template
# =========================================================
INDEX_HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>GenMatDesign-AI</title>
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f4f7fb;
            margin: 0;
            padding: 0;
            color: #222;
        }
        .container {
            max-width: 1180px;
            margin: 26px auto;
            background: #fff;
            border-radius: 18px;
            padding: 24px;
            box-shadow: 0 8px 28px rgba(0,0,0,0.08);
        }
        .hero {
            background: linear-gradient(135deg, #0d47a1, #1976d2);
            color: white;
            padding: 20px;
            border-radius: 16px;
            margin-bottom: 18px;
        }
        .hero p {
            margin: 8px 0 0 0;
            opacity: 0.95;
        }
        .status {
            padding: 12px;
            border-radius: 12px;
            background: #eef7ea;
            border: 1px solid #cfe8c6;
            margin-bottom: 18px;
        }
        .grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 18px;
        }
        .card {
            background: #fafcff;
            border: 1px solid #dde7f5;
            border-radius: 14px;
            padding: 16px;
        }
        h1, h2, h3 {
            margin-top: 0;
        }
        input, button {
            width: 100%;
            padding: 10px;
            margin-top: 8px;
            margin-bottom: 10px;
            border-radius: 10px;
            border: 1px solid #cad5e2;
            box-sizing: border-box;
            font-size: 14px;
        }
        button {
            background: #1459c7;
            color: white;
            border: none;
            cursor: pointer;
            font-weight: bold;
        }
        button:hover {
            background: #0f49a5;
        }
        .mono {
            font-family: Consolas, monospace;
            background: #f3f6fa;
            padding: 10px;
            border-radius: 8px;
            overflow-x: auto;
            white-space: pre-wrap;
        }
        img {
            max-width: 100%;
            border-radius: 12px;
            margin-top: 12px;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            margin-top: 12px;
        }
        th, td {
            border: 1px solid #dce5ef;
            padding: 8px;
            text-align: left;
            font-size: 14px;
            vertical-align: top;
        }
        th {
            background: #eef4fb;
        }
        .muted {
            color: #555;
        }
        .footer-note {
            margin-top: 14px;
            font-size: 13px;
            color: #666;
        }
        @media (max-width: 900px) {
            .grid { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body>
<div class="container">
    <div class="hero">
        <h1>GenMatDesign-AI</h1>
        <p>Generative AI for inverse materials design of solid-state battery electrolytes using a Conditional VAE, surrogate modeling, uncertainty estimation, and deployable Flask APIs.</p>
    </div>

    <div class="status">
        <b>Device:</b> {{ device }} |
        <b>Models Ready:</b> {{ ready }} |
        <b>Internal Library Size:</b> {{ rows }}
    </div>

    <div class="grid">
        <div class="card">
            <h2>1. Train Internal Model</h2>
            <p class="muted">Train the generator and surrogate model using the built-in electrolyte library.</p>
            <form method="post" action="/web/train">
                <button type="submit">Train Model</button>
            </form>
            {% if train_result %}
                <div class="mono">{{ train_result }}</div>
            {% endif %}
            {% if dataset_plot %}
                <img src="data:image/png;base64,{{ dataset_plot }}">
            {% endif %}
        </div>

        <div class="card">
            <h2>2. Predict One Electrolyte</h2>
            <p class="muted">Enter an electrolyte sequence such as <code>PEO-LiTFSI-SN-Al2O3</code>.</p>
            <form method="post" action="/web/predict">
                <input type="text" name="sequence" value="{{ sequence_value or 'PEO-LiTFSI-SN-Al2O3' }}">
                <button type="submit">Predict Properties</button>
            </form>
            {% if predict_result %}
                <div class="mono">{{ predict_result }}</div>
            {% endif %}
            {% if pred_plot %}
                <img src="data:image/png;base64,{{ pred_plot }}">
            {% endif %}
            {% if imp_plot %}
                <img src="data:image/png;base64,{{ imp_plot }}">
            {% endif %}
        </div>
    </div>

    <div class="card" style="margin-top:18px;">
        <h2>3. Inverse Design from Desired Properties</h2>
        <p class="muted">Specify target properties and generate new candidate electrolyte sequences ranked by closeness to target.</p>
        <form method="post" action="/web/generate">
            <div class="grid">
                <div>
                    <label>Ionic Conductivity</label>
                    <input type="text" name="ionic_conductivity" value="{{ gen_form.get('ionic_conductivity', '1.75') }}">
                </div>
                <div>
                    <label>Stability Window</label>
                    <input type="text" name="stability_window" value="{{ gen_form.get('stability_window', '5.15') }}">
                </div>
                <div>
                    <label>Glass Transition Temp</label>
                    <input type="text" name="glass_transition_temp" value="{{ gen_form.get('glass_transition_temp', '29') }}">
                </div>
                <div>
                    <label>Stability Score</label>
                    <input type="text" name="stability_score" value="{{ gen_form.get('stability_score', '0.90') }}">
                </div>
                <div>
                    <label>Top K</label>
                    <input type="text" name="top_k" value="{{ gen_form.get('top_k', '10') }}">
                </div>
                <div>
                    <label>Export</label>
                    <button formaction="/web/export" formmethod="get" type="submit">Download Last Generated CSV</button>
                </div>
            </div>
            <button type="submit">Generate Candidates</button>
        </form>

        {% if generated_candidates %}
        <table>
            <thead>
                <tr>
                    <th>#</th>
                    <th>Sequence</th>
                    <th>Distance</th>
                    <th>Predicted Properties</th>
                    <th>Uncertainty Std</th>
                </tr>
            </thead>
            <tbody>
                {% for row in generated_candidates %}
                <tr>
                    <td>{{ loop.index }}</td>
                    <td>{{ row["sequence"] }}</td>
                    <td>{{ "%.4f"|format(row["distance_to_target"]) }}</td>
                    <td><div class="mono">{{ row["predicted_properties"] }}</div></td>
                    <td><div class="mono">{{ row["uncertainty_std"] }}</div></td>
                </tr>
                {% endfor %}
            </tbody>
        </table>
        {% endif %}
    </div>

    <div class="grid" style="margin-top:18px;">
        <div class="card">
            <h2>4. Architecture</h2>
            <div class="mono">Built-in Electrolyte Library
        ↓
Tokenization + Vocabulary
        ↓
Conditional VAE Generator
        ↓
Surrogate Property Predictor
        ↓
Inverse Design from Target Properties
        ↓
Candidate Ranking + Uncertainty
        ↓
Dashboard + REST API</div>
        </div>

        <div class="card">
            <h2>5. API</h2>
            <div class="mono">GET  /api
GET  /health
POST /train
POST /predict
POST /generate
GET  /export</div>

            <h3>Example /predict payload</h3>
            <div class="mono">{
  "sequence": "PEO-LiTFSI-SN-Al2O3"
}</div>

            <h3>Example /generate payload</h3>
            <div class="mono">{
  "target": {
    "ionic_conductivity": 1.75,
    "stability_window": 5.15,
    "glass_transition_temp": 29.0,
    "stability_score": 0.90
  },
  "top_k": 10
}</div>
        </div>
    </div>

    {% if error %}
    <div class="card" style="margin-top:18px; border:1px solid #f1b7b7; background:#fff3f3;">
        <h2>Error</h2>
        <div class="mono">{{ error }}</div>
    </div>
    {% endif %}

    <div class="footer-note">
        This repository is a portfolio-grade inverse materials design prototype for GitHub. The internal library is illustrative and intended for demonstration, not publication-grade discovery.
    </div>
</div>
</body>
</html>
"""


# =========================================================
# Rendering helper
# =========================================================
def render_home(**kwargs):
    rows = len(GLOBAL["df"]) if GLOBAL["df"] is not None else len(load_internal_data())
    return render_template_string(
        INDEX_HTML,
        device=DEVICE,
        ready=is_ready(),
        rows=rows,
        train_result=kwargs.get("train_result"),
        dataset_plot=kwargs.get("dataset_plot"),
        predict_result=kwargs.get("predict_result"),
        pred_plot=kwargs.get("pred_plot"),
        imp_plot=kwargs.get("imp_plot"),
        generated_candidates=kwargs.get("generated_candidates"),
        sequence_value=kwargs.get("sequence_value"),
        gen_form=kwargs.get("gen_form", {}),
        error=kwargs.get("error"),
    )


# =========================================================
# API routes
# =========================================================
@app.route("/", methods=["GET"])
def home():
    return render_home()


@app.route("/api", methods=["GET"])
def api_root():
    return jsonify({
        "project": "GenMatDesign-AI",
        "description": "Generative AI for inverse materials design of solid-state electrolytes",
        "endpoints": ["/health", "/train", "/predict", "/generate", "/export"],
    })


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "device": DEVICE,
        "models_ready": is_ready(),
        "internal_rows": int(len(load_internal_data())),
    })


@app.route("/train", methods=["POST"])
def train_api():
    try:
        result = train_models()
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/predict", methods=["POST"])
def predict_api():
    try:
        if not is_ready():
            return jsonify({"error": "Models are not ready. Call /train first."}), 400

        payload = request.get_json(silent=True) or {}
        sequence = str(payload.get("sequence", "")).strip()
        if not sequence:
            return jsonify({"error": "Missing sequence"}), 400

        pred = predict_sequence(sequence)
        unc = predict_uncertainty(sequence)
        imp = token_importance(sequence)

        return jsonify({
            "sequence": sequence,
            "predicted_properties": pred,
            "uncertainty_std": unc["std"],
            "token_importance": imp,
        })
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/generate", methods=["POST"])
def generate_api():
    try:
        if not is_ready():
            return jsonify({"error": "Models are not ready. Call /train first."}), 400

        payload = request.get_json(silent=True) or {}
        target = payload.get("target", {})
        top_k = int(payload.get("top_k", 10))

        for c in TARGET_COLUMNS:
            if c not in target:
                return jsonify({"error": f"Missing target property: {c}"}), 400

        rows = generate_candidates(target, top_k=top_k)
        return jsonify({
            "target": target,
            "num_returned": len(rows),
            "candidates": rows,
        })
    except Exception as e:
        return jsonify({"error": str(e), "trace": traceback.format_exc()}), 500


@app.route("/export", methods=["GET"])
def export_api():
    if not GLOBAL["last_generated"]:
        return jsonify({"error": "No generated candidates available. Call /generate first."}), 400

    flat_rows = []
    for row in GLOBAL["last_generated"]:
        flat_row = {"sequence": row["sequence"], "distance_to_target": row["distance_to_target"]}
        for k, v in row["predicted_properties"].items():
            flat_row[f"pred_{k}"] = v
        for k, v in row["uncertainty_std"].items():
            flat_row[f"unc_{k}"] = v
        flat_rows.append(flat_row)

    df = pd.DataFrame(flat_rows)
    csv_data = df.to_csv(index=False)

    resp = make_response(csv_data)
    resp.headers["Content-Disposition"] = "attachment; filename=generated_candidates.csv"
    resp.headers["Content-Type"] = "text/csv"
    return resp


# =========================================================
# Web routes
# =========================================================
@app.route("/web/train", methods=["POST"])
def web_train():
    try:
        result = train_models()
        dataset_plot = plot_dataset_summary(GLOBAL["df"])
        return render_home(
            train_result=json.dumps(result, indent=2),
            dataset_plot=dataset_plot,
        )
    except Exception as e:
        return render_home(error=f"{str(e)}\n\n{traceback.format_exc()}")


@app.route("/web/predict", methods=["POST"])
def web_predict():
    try:
        if not is_ready():
            return render_home(error="Models are not ready. Train the model first.")

        sequence = request.form.get("sequence", "").strip()
        if not sequence:
            return render_home(error="Please enter a valid sequence.")

        pred = predict_sequence(sequence)
        unc = predict_uncertainty(sequence)["std"]
        imp = token_importance(sequence)

        result = {
            "sequence": sequence,
            "predicted_properties": pred,
            "uncertainty_std": unc,
            "token_importance": imp,
        }

        return render_home(
            predict_result=json.dumps(result, indent=2),
            pred_plot=plot_prediction(pred),
            imp_plot=plot_token_importance(imp),
            sequence_value=sequence,
        )
    except Exception as e:
        return render_home(error=f"{str(e)}\n\n{traceback.format_exc()}")


@app.route("/web/generate", methods=["POST"])
def web_generate():
    try:
        if not is_ready():
            return render_home(error="Models are not ready. Train the model first.")

        gen_form = {
            "ionic_conductivity": request.form.get("ionic_conductivity", "1.75"),
            "stability_window": request.form.get("stability_window", "5.15"),
            "glass_transition_temp": request.form.get("glass_transition_temp", "29"),
            "stability_score": request.form.get("stability_score", "0.90"),
            "top_k": request.form.get("top_k", "10"),
        }

        target = {
            "ionic_conductivity": float(gen_form["ionic_conductivity"]),
            "stability_window": float(gen_form["stability_window"]),
            "glass_transition_temp": float(gen_form["glass_transition_temp"]),
            "stability_score": float(gen_form["stability_score"]),
        }
        top_k = int(float(gen_form["top_k"]))

        rows = generate_candidates(target, top_k=top_k)

        return render_home(
            generated_candidates=rows,
            gen_form=gen_form,
        )
    except Exception as e:
        return render_home(error=f"{str(e)}\n\n{traceback.format_exc()}")


@app.route("/web/export", methods=["GET"])
def web_export():
    return export_api()


# =========================================================
# Main
# =========================================================
if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5045, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5045
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 12:08:27] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 12:08:27] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 12:08:38] "POST /web/train HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/Apr/2026 12:10:51] "GET / HTTP/1.1" 200 -
